In [2]:

# Notebook: Demo encuesta interactiva (versión completa y robusta)
# Pegar en una celda de Jupyter (VS Code) y ejecutar.
# Requisitos: pandas, openpyxl, ipywidgets

import os, glob, re
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets

# Helpers: Clase y funciones de carga
class EncuestaPregunta:
    def __init__(self, seccion, texto, tipo="Texto", opciones_raw=""):
        self.seccion = seccion or ""
        self.texto = texto or ""
        self.tipo = tipo or "Texto"
        self.opciones_raw = opciones_raw or ""

def load_from_xlsx_debug(path):
    """Lectura diagnóstica: detecta fila de header que contiene 'Pregunta', muestra primeras filas y construye lista de preguntas.
       Devuelve lista de diccionarios: {'seccion','texto','tipo','opciones'}"""
    print("Leyendo archivo:", path)
    raw = pd.read_excel(path, sheet_name=0, header=None, engine="openpyxl")
    print("Dimensiones (raw):", raw.shape)
    print("\n--- Primeras 12 filas crudas (mostrar columnas 0..6) ---")
    display(raw.iloc[:12, :7])
    # buscar fila que contenga 'Pregunta'
    header_row = None
    for i in range(min(40, raw.shape[0])):
        row_text = " ".join([str(x) for x in raw.iloc[i].values if pd.notna(x)])
        if re.search(r'pregunta', row_text, flags=re.I):
            header_row = i
            break
    print("Fila detectada como encabezado (index):", header_row)
    if header_row is None:
        print("No se detectó fila con 'Pregunta' — usando header=0 (fallback).")
        df = pd.read_excel(path, sheet_name=0, header=0, engine="openpyxl")
    else:
        df = pd.read_excel(path, sheet_name=0, header=header_row, engine="openpyxl")
    print("Dimensiones (df con header):", df.shape)
    print("Columnas detectadas:", list(df.columns))

    cols = list(df.columns)
    col_id = None
    col_preg = None
    col_opts = None
    # heurística para columnas
    for c in cols:
        cn = str(c).strip().lower()
        if cn == 'id' or cn.startswith('id'):
            col_id = c
        if 'pregunta' in cn:
            col_preg = c
        if 'opcion' in cn or 'respuesta' in cn or 'posibl' in cn:
            col_opts = c

    # fallbacks
    if col_preg is None and len(cols) >= 2:
        col_preg = cols[1]
        print("Fallback: usando segunda columna como 'Pregunta':", col_preg)
    if col_id is None:
        col_id = cols[0]
        print("Fallback: usando primera columna como 'ID':", col_id)
    if col_opts is None and len(cols) >= 3:
        col_opts = cols[2]
        print("Fallback: usando tercera columna como 'Opciones':", col_opts)

    # convertir IDs a numérico
    ids = pd.to_numeric(df[col_id], errors='coerce')
    print("IDs convertidos: valores únicos (muestra):", pd.Series(ids.dropna().unique())[:20].tolist())
    df_ids = df[~ids.isna()].copy()
    df_ids[col_id] = ids.dropna().astype(int)

    if df_ids.shape[0] == 0:
        print("No se encontraron filas con ID numérico. Tomando filas con pregunta no vacía.")
        non_empty = df[df[col_preg].notna() & (df[col_preg].astype(str).str.strip() != "")]
        print("Filas con pregunta no vacía:", non_empty.shape[0])
        display(non_empty.head(10).iloc[:, :6])
        df_unique = non_empty.reset_index(drop=True)
    else:
        df_unique = df_ids.drop_duplicates(subset=[col_id], keep='first').sort_values(by=col_id)
    preguntas = []
    for _, r in df_unique.iterrows():
        seccion = r.get('Seccion','') if 'Seccion' in r.index else ""
        texto = str(r.get(col_preg,"")).strip()
        opciones_raw = str(r.get(col_opts,"")).strip() if (col_opts in r.index) else ""
        tipo = "Seleccion" if '|' in opciones_raw else "Texto"
        preguntas.append({"seccion": seccion, "texto": texto, "tipo": tipo, "opciones": opciones_raw})
    print("Preguntas detectadas (len):", len(preguntas))
    print("Primeras 10 preguntas (previsualizacion):")
    for i,p in enumerate(preguntas[:10]):
        print(f"{i+1}. [{p['tipo']}] {p['texto'][:120]}")
    return preguntas

def load_from_csv(path):
    df = pd.read_csv(path, dtype=str).fillna("")
    preguntas = []
    # columnas esperadas: Seccion,Pregunta,Tipo,Opciones (si no existen, tomar primeras 2/3 columnas)
    cols = list(df.columns)
    col_sec = cols[0] if len(cols) >= 1 else None
    col_preg = cols[1] if len(cols) >= 2 else cols[0]
    col_tipo = cols[2] if len(cols) >= 3 else None
    col_opts = cols[3] if len(cols) >= 4 else None
    for _, row in df.iterrows():
        sec = row.get(col_sec,"") if col_sec in row.index else ""
        preg = row.get(col_preg,"")
        tipo = row.get(col_tipo,"Texto") if col_tipo in row.index else "Texto"
        opts = row.get(col_opts,"") if col_opts in row.index else ""
        preguntas.append(EncuestaPregunta(sec, preg, tipo, opts))
    return preguntas

# BÚSQUEDA AUTOMÁTICA DEL ARCHIVO Y CARGA (intenta localizar y cargar)
TARGET_NAMES = ["EncuestaAdmisiones.xlsx", "EncuestaAdmisiones.csv", "encuesta_admisiones.xlsx", "encuesta_demo.csv"]
found_paths = []

# rutas a revisar
candidates = [
    os.path.join(os.getcwd(), TARGET_NAMES[0]),
    os.path.join("/mnt/data", TARGET_NAMES[0]),
    os.path.join(os.path.expanduser("~"), TARGET_NAMES[0])
]
# búsqueda recursiva limitada
for root in ["/mnt", os.getcwd(), os.path.expanduser("~")]:
    try:
        for t in TARGET_NAMES:
            for path in glob.glob(os.path.join(root, "**", t), recursive=True):
                found_paths.append(path)
                if len(found_paths) >= 50: break
    except Exception:
        pass

# priorizar candidates existentes
for c in candidates:
    if os.path.exists(c) and c not in found_paths:
        found_paths.insert(0, c)

# unique
found_paths = list(dict.fromkeys(found_paths))

# Si encontramos, usar el primero; si no, mostrar uploader
preguntas_global = []
detected_path = None

if found_paths:
    detected_path = found_paths[0]
    print("Archivo encontrado automáticamente:", detected_path)
    if detected_path.lower().endswith(".xlsx"):
        pregs = load_from_xlsx_debug(detected_path)
        # convertir a clase EncuestaPregunta
        preguntas_global = [EncuestaPregunta(d.get('seccion',''), d.get('texto',''), d.get('tipo','Texto'), d.get('opciones','')) for d in pregs]
    else:
        # csv
        preguntas_global = load_from_csv(detected_path)
    print("\nAsignado preguntas_global con", len(preguntas_global), "elementos.")
else:
    print("No se encontró archivo automáticamente en rutas comunes.")
    # mostrar archivos del cwd para ayudar
    try:
        print("Archivos en cwd (primeros 200):", os.listdir(os.getcwd())[:200])
    except Exception:
        pass
    # uploader
    upload = widgets.FileUpload(accept=".xlsx, .csv", multiple=False)
    btn_upload = widgets.Button(description="Guardar archivo subido en cwd")
    out_upload = widgets.Output()

    def on_upload_save(b):
        with out_upload:
            clear_output()
            if not upload.value:
                print("No has subido ningún archivo aún.")
                return
            fname = list(upload.value.keys())[0]
            content = upload.value[fname]['content']
            save_path = os.path.join(os.getcwd(), fname)
            with open(save_path, "wb") as f:
                f.write(content)
            print("Archivo guardado en:", save_path)
            print("Reejecuta esta celda para que lo detecte automáticamente.")
    btn_upload.on_click(on_upload_save)
    display(widgets.HTML("<b>Sube tu EncuestaAdmisiones.xlsx o CSV aquí si no se detectó automáticamente:</b>"))
    display(upload, btn_upload, out_upload)

# Widgets principales y lógica segura
# estructura para respuestas en memoria: index -> valor
respuestas_guardadas = {}

# Creación segura de la lista de opciones para el Select
def build_options_from_preguntas():
    options = []
    for i, p in enumerate(preguntas_global):
        label = f"{i+1}. [{p.seccion}] {p.texto[:120]}"
        options.append((label, i))
    return options

# Widgets
list_box = widgets.Select(options=build_options_from_preguntas(), rows=15, layout=widgets.Layout(width='50%'))
resp_area = widgets.VBox([], layout=widgets.Layout(border='1px solid gray', padding='10px', width='100%'))
btn_prev = widgets.Button(description="⟨ Anterior")
btn_next = widgets.Button(description="Siguiente ⟩")
lbl_index = widgets.Label(value="Selecciona una pregunta")

# util para valores actuales
def _get_option_values():
    try:
        return [opt[1] for opt in list_box.options]
    except Exception:
        return []

# navegación segura
def go_prev(b):
    vals = _get_option_values()
    if not vals:
        return
    cur = list_box.value
    try:
        idx = vals.index(cur)
    except ValueError:
        idx = 0
    new_idx = max(0, idx - 1)
    list_box.value = vals[new_idx]

def go_next(b):
    vals = _get_option_values()
    if not vals:
        return
    cur = list_box.value
    try:
        idx = vals.index(cur)
    except ValueError:
        idx = 0
    new_idx = min(len(vals)-1, idx + 1)
    list_box.value = vals[new_idx]

btn_prev.on_click(go_prev)
btn_next.on_click(go_next)

def render_selected(change):
    val = change.get('new', None)
    if val is None:
        resp_area.children = [widgets.HTML("<i>No hay preguntas para mostrar.</i>")]
        lbl_index.value = "Selecciona una pregunta"
        return
    idx = int(val)
    p = preguntas_global[idx]
    title = widgets.HTML(f"<h4>{idx+1}. {p.texto}</h4><b>Sección:</b> {p.seccion}  &nbsp; <b>Tipo:</b> {p.tipo}")
    tipo_lower = str(getattr(p, "tipo", "")).lower()
    if "seleccion" in tipo_lower:
        opts = [o.strip() for o in str(getattr(p, "opciones_raw", "")).split("|") if o.strip()]
        control = widgets.RadioButtons(options=opts or ["Sí","No"])
    elif "escala" in tipo_lower:
        opts = [o.strip() for o in str(getattr(p, "opciones_raw", "")).split("|") if o.strip()]
        control = widgets.ToggleButtons(options=opts or ["1","2","3","4","5"])
    else:
        control = widgets.Textarea(placeholder="Escribe la respuesta aquí", layout=widgets.Layout(width='100%', height='120px'))
    # cargar valor guardado si existe
    if idx in respuestas_guardadas:
        try:
            control.value = respuestas_guardadas[idx]
        except Exception:
            pass
    resp_area.children = [title, control]
    lbl_index.value = f"Pregunta {idx+1} / {len(preguntas_global)}"

list_box.observe(lambda ch: render_selected(ch), names='value')

# Mostrar UI
display(widgets.HBox([list_box, resp_area]))
display(widgets.HBox([btn_prev, lbl_index, btn_next]))

# seleccionar primer elemento si existe
vals = _get_option_values()
if vals:
    list_box.value = vals[0]
else:
    resp_area.children = [widgets.HTML("<i>No hay preguntas para mostrar. Carga o sube el archivo y reejecuta esta celda.</i>")]

# Controles: guardar, estadísticas, recomendaciones, export
out_stats = widgets.Output()
out_rec = widgets.Output()
out_export = widgets.Output()

btn_save = widgets.Button(description="Guardar respuesta (pregunta actual)", button_style='success')
btn_stats = widgets.Button(description="Ver estadísticas", button_style='info')
btn_rec = widgets.Button(description="Generar recomendaciones", button_style='warning')
btn_export = widgets.Button(description="Exportar respuestas a CSV", button_style='')

lbl_save = widgets.HTML("")

def on_save(b):
    if list_box.value is None:
        lbl_save.value = "<span style='color:red'>Selecciona una pregunta primero.</span>"
        return
    idx = int(list_box.value)
    if len(resp_area.children) < 2:
        lbl_save.value = "<span style='color:orange'>No hay control de respuesta visible.</span>"
        return
    control = resp_area.children[1]
    val = ""
    try:
        val = control.value if hasattr(control, "value") else str(control)
    except Exception:
        val = str(getattr(control, "value", ""))
    respuestas_guardadas[idx] = str(val).strip()
    lbl_save.value = f"<span style='color:green'>Respuesta guardada (pregunta {idx+1}).</span>"

def on_stats(b):
    with out_stats:
        clear_output()
        if not respuestas_guardadas:
            print("No hay respuestas guardadas.")
            return
        rows = []
        for idx, val in respuestas_guardadas.items():
            p = preguntas_global[idx]
            rows.append({"Índice": idx+1, "Sección": p.seccion, "Pregunta": p.texto, "Respuesta": val})
        df = pd.DataFrame(rows)
        display(df)
        print("\nResumen por pregunta (conteos):")
        display(df.groupby(["Pregunta","Respuesta"]).size().reset_index(name="Conteo"))

def on_rec(b):
    with out_rec:
        clear_output()
        if not respuestas_guardadas:
            print("No hay respuestas guardadas para evaluar.")
            return
        rec_map = {}
        for idx, val in respuestas_guardadas.items():
            p = preguntas_global[idx]
            q = str(p.texto).lower()
            v = str(val).lower()
            if ("matem" in q or "álgebra" in q or "calculo" in q) and (v in ["bajo","1","2","baja","pobre"]):
                rec_map.setdefault("Matemáticas", []).extend([
                    "Curso básico de álgebra — PDF",
                    "Video: Fundamentos de Aritmética (playlist)",
                    "Guía de problemas resueltos - Nivel básico"
                ])
            if ("lect" in q or "lectura" in q or "comprensión" in q) and (v in ["bajo","1","2"]):
                rec_map.setdefault("Lectura crítica", []).extend([
                    "Taller de comprensión lectora — 5 sesiones",
                    "Práctica de resúmenes y esquemas",
                    "Manual de técnicas de estudio"
                ])
        if not rec_map:
            print("No se detectaron necesidades con las reglas demo.")
            return
        for tema, lista in rec_map.items():
            print(f"TEMA: {tema}")
            for it in lista:
                print(" -", it)
            print()

def on_export(b):
    with out_export:
        clear_output()
        if not respuestas_guardadas:
            print("No hay respuestas para exportar.")
            return
        rows = []
        for idx, val in respuestas_guardadas.items():
            p = preguntas_global[idx]
            rows.append({"Índice": idx+1, "Sección": p.seccion, "Pregunta": p.texto, "Respuesta": val})
        df = pd.DataFrame(rows)
        path = "respuestas_exportadas.csv"
        df.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"Respuestas exportadas a: {os.path.abspath(path)}")

btn_save.on_click(on_save)
btn_stats.on_click(on_stats)
btn_rec.on_click(on_rec)
btn_export.on_click(on_export)

display(widgets.HBox([btn_save, btn_stats, btn_rec, btn_export, lbl_save]))
display(out_stats, out_rec, out_export)

print("\nNotebook listo. Usa la lista a la izquierda para seleccionar preguntas, responde y pulsa 'Guardar respuesta'.")


KeyboardInterrupt: 